In [1]:
import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from scipy.special import softmax

import torch
from torch.utils.data import Dataset

import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

SEED = 42
MAX_LEN = 128

In [2]:
MODELS = [
    "allegro/herbert-base-cased",
    "dkleczek/bert-base-polish-cased-v1",
    "sdadas/polish-roberta-base-v2"
]

In [3]:
df = pd.read_csv("hate_train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

## Ważenie przykładów

In [4]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([0.5463, 5.8972])


In [5]:
from torchvision.ops import sigmoid_focal_loss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )


        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [6]:
import torch
import torch.nn.functional as F
from transformers import Trainer

class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


class FocalTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(
            alpha=0.75,
            gamma=2.0
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss = self.focal_loss(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [7]:

class HateDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    probs = softmax(pred.predictions, axis=1)[:, 1]
    preds = probs >= 0.5

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs)
    }

## Porównanie modeli

In [ ]:
results = []

for model_name in MODELS:

    print("=" * 50)
    print(model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

    val_dataset = HateDataset(
        val_df["sentence"],
        val_df["label"].values,
        tokenizer
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./tmp_{model_name.split('/')[-1]}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=20,
        report_to="none"
    )

    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate()

    results.append({
        "model": model_name,
        "f1": metrics["eval_f1"],
        "accuracy": metrics["eval_accuracy"]
    })

In [ ]:

results_df = pd.DataFrame(results)

print("\nRESULTS")
print(results_df.sort_values("f1", ascending=False))

best_model_name = results_df.sort_values(
    "f1",
    ascending=False
).iloc[0]["model"]

print(f"\nBEST MODEL: {best_model_name}")

## Analiza najlepszego modelu

In [8]:
# best_model_name = "dkleczek/bert-base-polish-cased-v1"
best_model_name = "allegro/herbert-base-cased"

In [9]:
print(best_model_name)

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

val_dataset = HateDataset(
    val_df["sentence"],
    val_df["label"].values,
    tokenizer
)

model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir=f"./best_{best_model_name.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = FocalTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()



allegro/herbert-base-cased


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/907k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,F1,Roc Auc,Pr Auc
0.012228,0.037016,5,0.923345,0.660000,0.194118,0.990756,0.300000,0.919736,0.540228


In [10]:
print(metrics)

{'eval_loss': 0.03701600804924965, 'eval_accuracy': 0.9233449477351916, 'eval_precision': 0.66, 'eval_recall': 0.19411764705882353, 'eval_specificity': 0.9907558455682436, 'eval_f1': 0.3, 'eval_roc_auc': 0.9197357899113969, 'eval_pr_auc': 0.5402279537483871}


In [11]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.99      0.96      1839
           1       0.66      0.19      0.30       170

    accuracy                           0.92      2009
   macro avg       0.80      0.59      0.63      2009
weighted avg       0.91      0.92      0.90      2009



In [12]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

print(cm_df)

          Predicted 0  Predicted 1
Actual 0         1822           17
Actual 1          137           33


## Najlepszy model

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

full_dataset = HateDataset(
    df["sentence"],
    df["label"].values,
    tokenizer
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir="./final_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=best_model,
    args=args,
    train_dataset=full_dataset
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

Step,Training Loss
500,0.264810
1000,0.178066
1500,0.142944
2000,0.099176
2500,0.074801
3000,0.045309


TrainOutput(global_step=3140, training_loss=0.13066622269381384, metrics={'train_runtime': 975.2716, 'train_samples_per_second': 51.478, 'train_steps_per_second': 3.22, 'total_flos': 2812176695789100.0, 'train_loss': 0.13066622269381384, 'epoch': 5.0})

## Predykcje

In [ ]:
with open("hate_test_data.txt", "r", encoding="utf8") as f:
    test_texts = [line.strip() for line in f]

enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

best_model.to(device)

enc = {
    k: v.to(device)
    for k, v in enc.items()
}

with torch.no_grad():
    outputs = best_model(**enc)

preds = outputs.logits.argmax(dim=1).cpu().numpy()

pd.DataFrame(preds).to_csv(
    "pred.csv",
    header=False,
    index=False
)
